# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv()

True

In [3]:
# TODO: Create a .env file with the following variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### VectorDB Instance

In [4]:
# Instantiate your ChromaDB Client
chroma_client = chromadb.PersistentClient(path="./chromadb")

### Collection

In [5]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_base="https://openai.vocareum.com/v1",
    api_key=OPENAI_API_KEY
)

In [7]:
# TODO: Create a collection
# Choose any name you want
collection = chroma_client.get_or_create_collection(
   name="udaplay",
   embedding_function=embedding_fn
)

### Add documents

In [15]:
# Make sure you have a directory "project/starter/games"
data_dir = "./games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # You can change what text you want to index
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [8]:
collection.count()

15

In [9]:
collection.peek(1)

{'ids': ['001'],
 'embeddings': array([[-0.00253281, -0.01467894, -0.00929119, ..., -0.020692  ,
         -0.00731421, -0.03772059]], shape=(1, 1536)),
 'documents': ['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': [{'Platform': 'PlayStation 1',
   'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
   'Name': 'Gran Turismo',
   'Publisher': 'Sony Computer Entertainment',
   'Genre': 'Racing',
   'YearOfRelease': 1997}]}

### Semantic Search Test

In [10]:
results = collection.query(
    query_texts=["Racing game on Playstation"],
    n_results=2,
)

In [11]:
results

{'ids': [['001', '003']],
 'embeddings': None,
 'documents': [['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
   '[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'YearOfRelease': 1997,
    'Platform': 'PlayStation 1',
    'Publisher': 'Sony Computer Entertainment',
    'Genre': 'Racing',
    'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
    'Name': 'Gran Turismo'},
   {'Name': 'Gran Turismo 5',
    'YearOfRelease': 2010,
    'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
    'Platform': 'PlayStation 3',
    '

---
**Remarks**:
Observing the result above, the semantic search performs well. The query asked for racing games on Playstation platform, and it indeed gave us racing games on Playstation.

### Vector Store Manager Test

In [12]:
from lib.vector_db import VectorStoreManager

In [13]:
manager = VectorStoreManager(
    openai_api_key=CHROMA_OPENAI_API_KEY,
    is_persistent=True,
    path="./chromadb"
)

In [14]:
store = manager.get_store("udaplay")

In [18]:
result = store.query(
    query_texts="Racing game on Playstation",
    n_results=2
)

In [19]:
result

{'ids': [['001', '003']],
 'embeddings': None,
 'documents': [['[PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.',
   '[PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.']],
 'uris': None,
 'included': ['documents', 'distances', 'metadatas'],
 'data': None,
 'metadatas': [[{'Platform': 'PlayStation 1',
    'Genre': 'Racing',
    'Name': 'Gran Turismo',
    'Publisher': 'Sony Computer Entertainment',
    'YearOfRelease': 1997,
    'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'},
   {'Publisher': 'Sony Computer Entertainment',
    'Name': 'Gran Turismo 5',
    'Description': 'A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.',
    'YearOfReleas

---
**Remarks**:
The VectorStoreManager library works well with the semantic search.